In [8]:
import polars as pl
import boto3
import io
from botocore.config import Config
from botocore import UNSIGNED
import os
import plotly.graph_objects as go
import importlib
import RateAcuity
importlib.reload(RateAcuity)
from RateAcuity import calculate_bill_gas, get_tariff_RA
import GenabilityHack
importlib.reload(GenabilityHack)
from GenabilityHack import calculate_bill_electric, get_tariff_gen
from copy import deepcopy

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

def download_aggregate_profile(state, upgrade=0):
    files = [ 
        f"up{upgrade:02}-{state.lower()}-mobile_home.csv",
        f"up{upgrade:02}-{state.lower()}-multi-family_with_2_-_4_units.csv",
        f"up{upgrade:02}-{state.lower()}-multi-family_with_5plus_units.csv",
        f"up{upgrade:02}-{state.lower()}-single-family_attached.csv",
        f"up{upgrade:02}-{state.lower()}-single-family_detached.csv"
    ]
    downloads = []
    for file in files:
        bucket = "oedi-data-lake"
        key = f"nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_tmy3_release_2/timeseries_aggregates/by_state/upgrade={upgrade}/state={state}/{file}"
        
        response = s3.get_object(Bucket=bucket, Key=key)
        buffer = io.BytesIO(response['Body'].read())
        cur = (pl.read_csv(buffer)
                        .select(["timestamp", "units_represented", "out.electricity.total.energy_consumption.kwh", "out.natural_gas.total.energy_consumption.kwh"])
                        .filter(pl.col("timestamp").str.contains("2018-"))
                        .with_columns(pl.col("timestamp").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                                      .dt.replace(year=2025)
                                      .alias("timestamp"))
        )

        downloads.append(cur)

    final = (pl.concat(downloads).with_columns(pl.col("timestamp").dt.truncate("1h").dt.strftime("%Y-%m-%dT%H:%M:%S.%f").alias("timestamp"))
                .group_by(["timestamp"])
                .agg([
                    pl.col("units_represented").first().alias("units_represented"),
                    pl.col("out.electricity.total.energy_consumption.kwh").sum().alias("electricity.total"),
                    pl.col("out.natural_gas.total.energy_consumption.kwh").sum().alias("natural_gas.total")
                ])
            ).sort(["timestamp"])

    final.write_csv(f"state_consumption_agg/{state}-{upgrade}.csv")
    return final

In [9]:
state = "MN"
utility = "Northern States Power Company"
gas_utility = "Northern States Power Company"
upgrade = 3
penetration = 0.3 # Assumes this factor of gas customers upgrade

# Get gas customer count, which is needed for accurate adjustment of number of customers
temp = pl.read_excel("c:/Users/al.qarooni/OneDrive - RMI/Documents/VSCode/Combination_Utility_Scan/Landscape_Scan.xlsx",sheet_name="Summary_working",columns=list(range(20)),has_header=False)
headers = temp[1].to_dicts()[0]
wanted = ["Gas Company", "State", "Overlap Percentage", "Total Customers (Gas)", "Total Customers (All - Est)"]
rename_map = {k: v for k, v in headers.items() if v in wanted}
temp = temp.slice(2).rename(rename_map).select(wanted).filter((pl.col("State")==state)&(pl.col("Gas Company")==gas_utility))
try:
    C_overlap = round(int(temp["Total Customers (All - Est)"].item()) * float(temp["Overlap Percentage"].item().replace("%","")))
    C_gas = int(temp["Total Customers (Gas)"].item())
except:
    raise Exception(f"Gas utility name '{gas_utility}' invalid. Valid options are: {temp['Gas Company'].to_list()}")
del temp, headers, wanted

# Get gas sales and convert to therms
temp = pl.read_excel("c:/Users/al.qarooni/OneDrive - RMI/Documents/VSCode/Combination_Utility_Scan/Landscape_Scan.xlsx",sheet_name="Data_by_State",has_header=False)
headers = temp[1].to_dicts()[0]
wanted = ["Company", "State of Operation", "Residential Natural Gas Volume (Dth)"]
rename_map = {k: v for k, v in headers.items() if v in wanted}
temp = temp.slice(2).rename(rename_map).select(wanted).filter((pl.col("State of Operation")==state)&(pl.col("Company")==gas_utility))
gas_sales = int(temp["Residential Natural Gas Volume (Dth)"].item())*10/0.03412

if os.path.exists(f"state_consumption_agg/{state}-0.csv"):
    base = pl.read_csv(f"state_consumption_agg/{state}-0.csv")
else:
    base = download_aggregate_profile(state)

if os.path.exists(f"state_consumption_agg/{state}-{upgrade}.csv"):
    new = pl.read_csv(f"state_consumption_agg/{state}-{upgrade}.csv")
else:
    new = download_aggregate_profile(state, upgrade)

#Just to make sure
assert base.columns==new.columns and base.shape==new.shape

state_consumption = pl.DataFrame(base["timestamp"])
state_consumption = state_consumption.with_columns(
    (1-penetration)*base.drop('timestamp') + penetration*new.drop('timestamp')
)

In [10]:
def monthly_avg(df):
    return (
        df.with_columns(pl.col("timestamp").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S%.f"))
          .with_columns(month=pl.col("timestamp").dt.strftime("%Y-%m"))
          .group_by("month")
          .mean()
          .sort("month")
    )

# Convert to pandas
base_df = monthly_avg(base.select(["timestamp", "electricity.total", "natural_gas.total"])).to_pandas()
new_df = monthly_avg(new.select(["timestamp", "electricity.total", "natural_gas.total"])).to_pandas()
blended_df = monthly_avg(state_consumption.select(["timestamp", "electricity.total", "natural_gas.total"])).to_pandas()

# Plot
fig = go.Figure()

# Electricity (left y-axis)
fig.add_trace(go.Scatter(x=base_df["month"], y=base_df["electricity.total"],
                         name="Electricity - Baseline", yaxis="y1", mode="lines"))
fig.add_trace(go.Scatter(x=blended_df["month"], y=blended_df["electricity.total"],
                         name=f"Electricity - {int(penetration*100)}% Upgrade", yaxis="y1", mode="lines"))
fig.add_trace(go.Scatter(x=new_df["month"], y=new_df["electricity.total"],
                         name="Electricity - 100% Upgrade", yaxis="y1", mode="lines"))

# Gas (right y-axis)
fig.add_trace(go.Scatter(x=base_df["month"], y=base_df["natural_gas.total"],
                         name="Gas - Baseline", yaxis="y2", line=dict(dash='dot'), mode="lines"))
fig.add_trace(go.Scatter(x=blended_df["month"], y=blended_df["natural_gas.total"],
                         name=f"Gas - {int(penetration*100)}% Upgrade", yaxis="y2", line=dict(dash='dot'), mode="lines"))
fig.add_trace(go.Scatter(x=new_df["month"], y=new_df["natural_gas.total"],
                         name="Gas - 100% Upgrade", yaxis="y2", line=dict(dash='dot'), mode="lines"))

fig.update_layout(
    title="Monthly Energy Use: Electricity vs Gas",
    xaxis=dict(title="Month"),
    yaxis=dict(title="Electricity Use", side="left"),
    yaxis2=dict(title="Gas Use", overlaying="y", side="right"),
    template="plotly_white",
    legend=dict(orientation="h", y=-0.2),
    height=600
)

fig.show()


In [11]:
tariff_list = pl.read_csv(f"RateData/{utility}.csv")

In [12]:
## BASELINE LOADS AND CUSTOMER NUMBERS COLLECTED BY RATE CLASS
# Normalize electric loads for per_tariff load prep and adjust natural gas according to reported
normalized_base_profile = base.with_columns([
    (pl.col("electricity.total") / pl.col("electricity.total").sum()).alias("normalized_elec_consumption"),
    (gas_sales * pl.col("natural_gas.total") / pl.col("natural_gas.total").sum()).alias("natural_gas.total")
])

# Build the service_schedules dict
service_schedules_orig = {}

for tariff in tariff_list.iter_rows(named=True):
     if tariff.get("Sales"):
        master_tariff_id = str(tariff["masterTariffId"])
        tariff_sales = tariff["Sales"]
        num_customers = tariff["Customers"]

        # Scale normalized profile to match utility reported tariff sales (added in tariff list json from rate case in previous cell) in kwh
        tariff_df = normalized_base_profile.select([
            pl.col("timestamp"),
            (pl.col("normalized_elec_consumption") * tariff_sales * 1000).alias("electricity.total")
        ]).with_columns(
            pl.lit(num_customers).alias("units_represented"),
        )

        # Store in dict
        service_schedules_orig[(master_tariff_id, tariff["Rate"])] = tariff_df

In [13]:
## POST-PENTRATION LOADS AND CUSTOMERS BY RATE CLASS
normalized_pen_profile = state_consumption.with_columns([
    (pl.col("electricity.total") / pl.col("electricity.total").sum()).alias("normalized_elec_consumption"),
    (gas_sales * pl.col("natural_gas.total").sum()/base["natural_gas.total"].sum() * 
     pl.col("natural_gas.total") / pl.col("natural_gas.total").sum()).alias("natural_gas.total")
])

# Build the service_schedules dict
service_schedules_with_penetration = {}

for tariff in tariff_list.iter_rows(named=True):
     if tariff.get("Sales"):
        master_tariff_id = str(tariff["masterTariffId"])
        if "From" in str(tariff["From/To"]):
            tariff_sales = tariff["Sales"]
            # Assuming all upgraded homes were served electricity by same utility
            num_customers=tariff["Customers"] - int(C_overlap*penetration)
        if "To" in str(tariff["From/To"]):
            # Add all additional load to target tariff sales
            tariff_sales = tariff["Sales"] + tariff_list["Sales"].sum() * (state_consumption["electricity.total"].sum()-base["electricity.total"].sum())/base["electricity.total"].sum()
            num_customers=tariff["Customers"] + int(C_overlap*penetration)
        if "From" not in str(tariff["From/To"]) and "To" not in str(tariff["From/To"]):
            tariff_sales = tariff["Sales"]
            num_customers = tariff["Customers"]

        # Scale normalized profile to match utility reported tariff sales (added in tariff list json from rate case in previous cell) in kwh
        tariff_df = normalized_pen_profile.select([
            pl.col("timestamp"),
            (pl.col("normalized_elec_consumption") * tariff_sales * 1000).alias("electricity.total")
        ]).with_columns(
            pl.lit(num_customers).alias("units_represented")
        )

        # Store in dict
        service_schedules_with_penetration[(master_tariff_id, tariff["Rate"])] = tariff_df

In [14]:
import requests

# app_id = "3df8e135-968d-4399-9879-2a1c6a3de30c"
# app_key = "e51974c7-996b-4698-9628-71950d223364"

# # API call to Genability for Elec Revenue Calc
# url = "https://api.genability.com/rest/v1/ondemand/calculate"
base_elec_data={}
tariffs={}
pen_elec_data={}
for (tariffId, base_consumption),(_,pen_consumption) in zip(service_schedules_orig.items(),service_schedules_with_penetration.items()):
    customers = base_consumption["units_represented"][0]
    per_cust = base_consumption.with_columns([
        (pl.col("electricity.total")/customers).alias("electricity.total")
    ])
    tariff = get_tariff_gen(tariffId[0],utility,"55429",per_cust)
    tariffs[tariffId[0]]=tariff
    revenue = calculate_bill_electric(tariff,per_cust)
    base_elec_data[tariffId] = {k: v * customers for k,v in revenue.items()}
    # params = {
    #     "masterTariffId": tariffId[0],
    #     "fromDateTime": base_consumption["timestamp"].first(),
    #     "toDateTime": base_consumption["timestamp"].last(),
    #     "detailLevel": "ALL",
    #     "propertyInputs": [{
    #         "keyName": "consumption",
    #         "unit": "kWh",
    #         "fromDateTime": base_consumption["timestamp"].first(),
    #         "duration": 3600000, # 1 hour
    #         "dataSeries": base_consumption["electricity.total"].to_list()
    #     }]
    # }
    # response = requests.post(url, auth=(app_id, app_key), json=params)
    # base_elec_data[tariffId] = response.json()["results"][0]
    base_elec_data[tariffId]["Customers"] = customers

    customers = pen_consumption["units_represented"][0]
    per_cust = pen_consumption.with_columns([
        (pl.col("electricity.total")/customers).alias("electricity.total")
    ])
    revenue = calculate_bill_electric(tariff,per_cust)
    pen_elec_data[tariffId] = {k: v * customers for k,v in revenue.items()}
    # params = {
    #     "masterTariffId": tariffId[0],
    #     "fromDateTime": pen_consumption["timestamp"].first(),
    #     "toDateTime": pen_consumption["timestamp"].last(),
    #     "detailLevel": "ALL",
    #     "propertyInputs": [{
    #         "keyName": "consumption",
    #         "unit": "kWh",
    #         "fromDateTime": pen_consumption["timestamp"].first(),
    #         "duration": 3600000, # 1 hour
    #         "dataSeries": pen_consumption["electricity.total"].to_list()
    #     }]
    # }
    # response = requests.post(url, auth=(app_id, app_key), json=params)
    # base_elec_data[tariffId] = response.json()["results"][0]
    pen_elec_data[tariffId]["Customers"] = customers

# RateAcuity call for gas revenue calc
tariff_gas = get_tariff_RA(state,gas_utility,"101-RESIDENTIAL FIRM SERVICE---")
customers = C_gas
per_cust = normalized_base_profile.with_columns([
    (pl.col("natural_gas.total")/customers).alias("natural_gas.total")
])
base_gas_data = {k: v*customers for k,v in calculate_bill_gas(tariff_gas, per_cust).items()}
customers = (C_gas-round(C_overlap*penetration))
per_cust = normalized_pen_profile.with_columns([
    (pl.col("natural_gas.total")/customers).alias("natural_gas.total")
])
pen_gas_data = {k: v*customers for k,v in calculate_bill_gas(tariff_gas, per_cust).items()}

In [15]:
# Collecting and printing totals
from collections import defaultdict
base_elec_revenues, pen_elec_revenues = deepcopy(base_elec_data), deepcopy(pen_elec_data)
for data in [base_elec_revenues,pen_elec_revenues]:
    rows = []
    for id, d in data.items():
        data[id]["TOTAL"]=0
        orig_fixed_price = 0
        tariff = tariffs[id[0]]
        for k,v in d.items():
            if not tariff.filter((pl.col("rateName")==k) & (pl.col("Rate Determinant").str.contains_any(["bill","day","month","year"]))).is_empty():
                orig_fixed_price+=data[id][k] if "Customer" in k else 0
            data[id]["TOTAL"]+= data[id][k] if k!="TOTAL" else 0
        data[id]["fixed_price"] = orig_fixed_price/data[id]["Customers"]/12

base_gas_revenue = 0
pen_gas_revenue = 0
for (_,v1), (_,v2) in zip(base_gas_data.items(),pen_gas_data.items()):
    base_gas_revenue += v1
    pen_gas_revenue += v2

print(f"""
    BASE:
    Electric Revenue: ${sum([b["TOTAL"] for b in base_elec_revenues.values()]):,.0f}
    Gas Revenue: ${base_gas_revenue:,.0f}
    Total: ${sum([b["TOTAL"] for b in base_elec_revenues.values()]) + base_gas_revenue:,.0f}
""")
print(f"""
    {int(penetration*100)}% PENETRATION OF UPGRADE {upgrade}:
    Electric Revenue: ${sum([p["TOTAL"] for p in pen_elec_revenues.values()]):,.0f}
    Gas Revenue: ${pen_gas_revenue:,.0f}
    Total: ${sum([p["TOTAL"] for p in pen_elec_revenues.values()]) + pen_gas_revenue:,.0f}
""")


    BASE:
    Electric Revenue: $1,452,249,375
    Gas Revenue: $379,228,564
    Total: $1,831,477,939


    30% PENETRATION OF UPGRADE 3:
    Electric Revenue: $1,685,798,497
    Gas Revenue: $286,193,044
    Total: $1,971,991,541



In [16]:
fun = lambda x: int(tariff_list.filter(pl.col("masterTariffId")==int(x))["Revenues"].item().replace("$","").replace(",","").replace(".00 ",""))
[
    (
        id,
        f"{b['Customers']} Customers",
        f"${b['TOTAL']/1000:,.0f}.00, {100*b['TOTAL']/sum([b['TOTAL'] for b in base_elec_revenues.values()]):.2f}% (calculated)",
        f"${fun(id[0]):,.0f}.00, {100*fun(id[0])/tariff_list[0]['Total Revenues'].item():.2f}% (reported)",
        f"Per customer difference ${(b['TOTAL'] - fun(id[0])*1000)/b['Customers']:,.2f}"
    )
    for id,b in base_elec_revenues.items()
]

[(('698', 'Residential'),
  '1165709 Customers',
  '$1,358,230.00, 93.53% (calculated)',
  '$1,295,489.00, 93.22% (reported)',
  'Per customer difference $53.82'),
 (('700', 'Residential - Electric Heat'),
  '44180 Customers',
  '$53,815.00, 3.71% (calculated)',
  '$52,457.00, 3.77% (reported)',
  'Per customer difference $30.75'),
 (('3324564', 'Residential - Water Heating'),
  '44 Customers',
  '$26.00, 0.00% (calculated)',
  '$24.00, 0.00% (reported)',
  'Per customer difference $52.76'),
 (('703', 'Residential - Time of Day'),
  '816 Customers',
  '$1,148.00, 0.08% (calculated)',
  '$1,234.00, 0.09% (reported)',
  'Per customer difference $-105.22'),
 (('704', 'Residential - Time of Day, Electric Heat'),
  '95 Customers',
  '$180.00, 0.01% (calculated)',
  '$94.00, 0.01% (reported)',
  'Per customer difference $909.63'),
 (('708', 'Limited Off Peak Service - Residential'),
  '363 Customers',
  '$144.00, 0.01% (calculated)',
  '$252.00, 0.02% (reported)',
  'Per customer difference 

In [ ]:
## RATE DESIGN CELL - ALL YEAR
select = ('698', 'Residential')
original_revenue = base_elec_revenues[select]["TOTAL"]
customers = base_elec_revenues[select]["Customers"]
orig_fixed_price = base_elec_revenues[select]["fixed_price"]

# Select the target tariff
tariff = tariffs[select[0]]

# Apply volumetric adjustment
volumetric_change = 0.9
tariff = tariff.with_columns(
    pl.when(pl.col("rateName").str.contains_any(["Winter","Summer"]))
        .then(pl.col("Rate")*volumetric_change)
    .otherwise(pl.col("Rate"))
    .alias("Rate")
)
per_cust = service_schedules_orig[select].with_columns([
    (pl.col("electricity.total")/customers).alias("electricity.total")
])
tariff_revenue = calculate_bill_electric(tariff, per_cust)
tariff_revenue = {k:v*customers for k,v in tariff_revenue.items()}

# Find new fixed price with revenue neutrality
new_non_fixed = sum([rev for key,rev in tariff_revenue.items() if "Customer" not in key])
fixed_total = original_revenue - new_non_fixed
factor = fixed_total/sum([rev for key,rev in tariff_revenue.items() if "Customer" in key])
tariff_revenue["TOTAL"] = 0
tariff_revenue["fixed_price"] = 0
for key, val in tariff_revenue.items():
    if "Customer" in key:
        tariff_revenue[key]*=factor
        tariff_revenue["fixed_price"]+=tariff_revenue[key] if "Customer" in key else 0
    tariff_revenue["TOTAL"] += tariff_revenue[key] if all([key!=sub for sub in ["fixed_price","TOTAL"]]) else 0

# Compute new monthly fixed price per customer and print
tariff_revenue["fixed_price"] = tariff_revenue["fixed_price"] / customers / 12

print(select)
print(f"Reducing volumetric rates by {(1 - volumetric_change) * 100:.2f}%, with new monthly customer charge of ${round(tariff_revenue['fixed_price'], 2)} (prev ${round(orig_fixed_price,2)})")
tariff_revenue

('698', 'Residential')
Reducing volumetric rates by 20.00%, with new monthly customer charge of $19.73 (prev $6.0)


{'Affordability Surcharge': 27371368.01889612,
 'Conservation Improvement Program Adjustment': -3146934.533999998,
 'Customer Charge': 275981701.51389027,
 'Fuel Cost Charge': 211656344.91203985,
 'Renewable Development Fund': 9840197.324219996,
 'Renewable Energy Standard Adjustment': 33712304.28,
 'Summer Rate': 215230984.6921819,
 'Transmission Cost Recovery Charge': 39275360.945519984,
 'Winter Rate': 548308793.3633798,
 'TOTAL': 1358230120.516128,
 'fixed_price': 19.729173512564046}

In [23]:
base_elec_revenues[select]

{'Affordability Surcharge': 27371368.01889612,
 'Conservation Improvement Program Adjustment': -3146934.5339999977,
 'Customer Charge': 83931048.0,
 'Fuel Cost Charge': 211656344.91203988,
 'Renewable Development Fund': 9840197.324219994,
 'Renewable Energy Standard Adjustment': 33712304.28,
 'Summer Rate': 269038730.86522734,
 'Transmission Cost Recovery Charge': 39275360.945519984,
 'Winter Rate': 685385991.704225,
 'Customers': 1165709,
 'TOTAL': 1358230120.516128,
 'fixed_price': 6.0}

In [26]:
## RATE DESIGN CELL - SEASONAL
original_revenue = base_elec_revenues[select]["TOTAL"]
customers = base_elec_revenues[select]["Customers"]
orig_fixed_price = base_elec_revenues[select]["fixed_price"]

# Select the target tariff
tariff = tariffs[select[0]]

# Apply volumetric adjustment to winter season
volumetric_change = 0.8
tariff = tariff.with_columns(
    pl.when(pl.col("rateName").str.contains_any(["Winter"]))
        .then(pl.col("Rate")*volumetric_change)
    .otherwise(pl.col("Rate"))
    .alias("Rate")
)
per_cust = service_schedules_orig[select].with_columns([
    (pl.col("electricity.total")/customers).alias("electricity.total")
])
tariff_revenue = calculate_bill_electric(tariff, per_cust)
tariff_revenue = {k:v*customers for k,v in tariff_revenue.items()}

# Find new fixed price with revenue neutrality
new_non_fixed = sum([rev for key,rev in tariff_revenue.items() if "Customer" not in key])
fixed_total = original_revenue - new_non_fixed
factor = fixed_total/sum([rev for key,rev in tariff_revenue.items() if "Customer" in key])
tariff_revenue["TOTAL"] = 0
tariff_revenue["fixed_price"] = 0
for key, val in tariff_revenue.items():
    if "Customer" in key:
        tariff_revenue[key]*=factor
        tariff_revenue["fixed_price"]+=tariff_revenue[key]
    tariff_revenue["TOTAL"] += tariff_revenue[key] if all([key!=sub for sub in ["fixed_price","TOTAL"]]) else 0

# Compute new monthly fixed price per customer and print
tariff_revenue["fixed_price"] = tariff_revenue["fixed_price"] / customers / 12

print(select)
print(f"Reducing winter volumetric rates by {(1 - volumetric_change) * 100:.2f}%, with new monthly customer charge of ${round(tariff_revenue['fixed_price'], 2)} (prev ${round(orig_fixed_price,2)})")
tariff_revenue

('698', 'Residential')
Reducing winter volumetric rates by 20.00%, with new monthly customer charge of $15.88 (prev $6.0)


{'Affordability Surcharge': 27371368.01889612,
 'Conservation Improvement Program Adjustment': -3146934.5339999977,
 'Customer Charge': 222173955.34084514,
 'Fuel Cost Charge': 211656344.91203985,
 'Renewable Development Fund': 9840197.324219994,
 'Renewable Energy Standard Adjustment': 33712304.28,
 'Summer Rate': 269038730.86522734,
 'Transmission Cost Recovery Charge': 39275360.94551998,
 'Winter Rate': 548308793.3633798,
 'TOTAL': 1358230120.516128,
 'fixed_price': 15.882605588876608}